# Scientific RAG from PDFs: A Transparent, Package-Native Walkthrough

## About This Notebook

This notebook demonstrates a **Graph-Augmented Retrieval-Augmented Generation (GraphRAG)** pipeline applied to a corpus of scientific papers. It is designed as a quality education tool — every data transformation is made visible, every package choice is explained, and prompt engineering decisions are treated as key tutorial content.

### The Domain Problem

Scientific literature in precision oncology and rare disease genomics grows faster than any researcher can read. Key findings about fusion transcript detection, gene–disease associations, and computational methods are scattered across dozens of papers. This pipeline offers a principled way to:

- **Search by meaning** (not just keywords) across a personal publication corpus
- **Ground LLM answers** in retrieved evidence, reducing hallucination
- **Build a knowledge graph** from the full corpus and use it to expand retrieval beyond what vector similarity alone finds

### Pipeline Overview

The notebook runs in two phases:

**Phase 1 — Build the indices (run once)**
```
PDF corpus
  → Text Extraction       (PyMuPDF4LLM)
  → Document Wrapping     (LlamaIndex Document)
  → Chunking              (SentenceSplitter)
  → Embeddings            (Sentence Transformers)   ← vector index
  → Full-corpus triplet extraction  (GPT-4o-mini)
  → Junk filter           (rule-based)
  → Entity + predicate normalisation  (GPT-4o-mini)
  → Knowledge Graph       (NetworkX)                ← graph index
```

**Phase 2 — Query (run per question)**
```
User question
  → Query Rewriting       (GPT-4o-mini)            ← Prompt Engineering Step 1
  → Vector Retrieval      (cosine similarity)
  → Context Assembly                                 ← Prompt Engineering Step 2
  → Grounded Answer       (GPT-4.1)                 ← Prompt Engineering Step 3
  → Graph-Augmented Retrieval  (entity expansion)
  → Enriched Answer       (GPT-4.1)
  → Side-by-side comparison
```

### Reading Guide

Each major section follows the same pattern:

> **Input →** what object goes in  
> **Method →** which package/technique is applied, and why  
> **Output →** what object comes out  

Code is written inline.  There are no hidden helper functions to obfuscate the analytical process. Every output is inspected explicitly.

## 1. Environment Setup

**Input →** system environment, a `.env` file containing `OPENAI_API_KEY`  
**Method →** standard library imports + `python-dotenv` to load secrets from `.env` without hardcoding them  
**Output →** a fully configured Python environment with a verified API key and a ready `outputs/` directory

We will import all dependencies at the beginning so that the full package requirements are visible at a glance.  Additions versus a minimal RAG pipeline: `networkx` for graph construction and `matplotlib` for visualization.

### Package Reference

Each package below appears later in the notebook. One sentence on what it does and why it was chosen over the common alternative:

| Package | What it does | Why this one |
|---|---|---|
| `pymupdf4llm` | Converts PDF pages to Markdown, preserving headings, tables, and lists | Produces structured Markdown rather than collapsed plain text; better than `pdfplumber` or `pdfminer` for scientific papers where section structure matters |
| `llama_index` | Provides `Document` and `SentenceSplitter` abstractions for wrapping and chunking text | `SentenceSplitter` respects sentence boundaries and automatically propagates metadata to child chunks; simpler than wiring up LangChain's `RecursiveCharacterTextSplitter` for this use case |
| `sentence_transformers` | Loads and runs the `all-MiniLM-L6-v2` embedding model locally | Runs on CPU without an API key; strong semantic quality for English scientific text; faster than calling an embeddings API for a corpus of this size |
| `sklearn` | Computes cosine similarity between query and chunk embeddings in one matrix operation | `cosine_similarity` from `sklearn.metrics.pairwise` is vectorised and runs in milliseconds over hundreds of chunks; no need for a dedicated vector database at this scale |
| `networkx` | Builds and traverses the knowledge graph | Pure-Python, no compilation required; `DiGraph` and path-finding functions are sufficient for a single-machine corpus; alternatives like `igraph` or `graph-tool` are faster at scale but require native extensions |
| `matplotlib` | Renders the graph visualisation | Standard scientific plotting library; used here only for the static graph image; an interactive alternative is `pyvis` |
| `python-dotenv` | Loads `OPENAI_API_KEY` from a `.env` file | Keeps secrets out of notebook source; the conventional alternative is setting environment variables manually, which is fragile across sessions |


In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

import pymupdf4llm

from dotenv import load_dotenv
load_dotenv()

api_key_present = os.environ.get("OPENAI_API_KEY") is not None
print(f"OpenAI API key loaded: {api_key_present}")
if not api_key_present:
    raise EnvironmentError("OPENAI_API_KEY not found. Add it to your .env file before continuing.")

from openai import OpenAI
client = OpenAI()

# Ensure the outputs directory exists for saving artefacts
Path('outputs').mkdir(exist_ok=True)
print("outputs/ directory ready")

## 2. Discover the PDF Corpus

**Input →** a directory of PDF files (`data/papers/`)  
**Method →** `pathlib.Path.glob('*.pdf')` — a zero-dependency way to enumerate files by extension; returns `Path` objects that carry both the filename stem (our `paper_id`) and the full path  
**Output →** `corpus_df`: a DataFrame with one row per paper, serving as the stable index for the entire pipeline

> **What to notice:** The `paper_id` column (the filename without extension) will travel through every downstream stage as a provenance key. When we eventually retrieve a passage or extract a triplet, we'll always know which paper it came from.

In [ ]:
data_dir = Path('data/papers')

if not data_dir.exists() or not list(data_dir.glob('*.pdf')):
    raise FileNotFoundError(
        f"No PDFs found in '{data_dir}'. "
        "Create the directory and add your papers there before continuing."
    )

pdf_files = sorted(data_dir.glob('*.pdf'))

corpus_df = pd.DataFrame({
    'paper_id': [p.stem for p in pdf_files],
    'path': [str(p) for p in pdf_files],
})

print(f"Found {len(corpus_df)} PDFs\n")
corpus_df

## 3. Extract Text with PyMuPDF4LLM

**Input →** `corpus_df` — file paths to PDFs  
**Method →** `pymupdf4llm.to_markdown()` — converts each PDF page to Markdown, preserving structure such as headings, tables, and lists. This is significantly better than raw text extraction for scientific papers, where structure carries meaning.  
**Output →** `docs_df` — adds `text` (a Markdown string) and `char_count` columns to the corpus table

> **Why Markdown output?** Scientific papers have section headings (`## Methods`), figure captions, and tables. PyMuPDF4LLM's Markdown output preserves these as `##` headings and `|` pipe-delimited tables, which downstream chunkers can respect. Plain-text extraction collapses this structure entirely, making it harder for the LLM to interpret retrieved passages in context.

> **What to notice:** Check the `char_count` column — large variance between papers is normal (a 3-page letter vs. a 12-page methods paper). The sample below shows the first 1000 characters of the first extracted paper so you can inspect the raw Markdown quality.

In [ ]:
records = []
for pdf_path in pdf_files:
    text = pymupdf4llm.to_markdown(str(pdf_path))
    records.append({
        'paper_id': pdf_path.stem,
        'path': str(pdf_path),
        'text': text,
        'char_count': len(text)
    })

docs_df = pd.DataFrame(records)

print(f"Extracted text from {len(docs_df)} papers\n")
print(docs_df[['paper_id', 'char_count']].to_string(index=False))

print(f"\n--- Sample: first 1000 characters of '{docs_df.iloc[0]['paper_id']}' ---\n")
print(docs_df.iloc[0]['text'][:1000])

## 4. Wrap into LlamaIndex Documents

**Input →** `docs_df` — raw Markdown text strings paired with paper metadata  
**Method →** `llama_index.core.Document` — a lightweight container that binds text to a metadata dictionary. LlamaIndex's downstream tools (the `SentenceSplitter` in the next step) expect this type and automatically propagate metadata to child chunks.  
**Output →** `documents`: a `List[Document]`, one per paper, each carrying `paper_id` and `source_path` in `.metadata`

> **Why not just pass raw strings to the chunker?** The `Document` wrapper is the handshake between extraction and chunking. When a document is split into chunks, each chunk automatically inherits its parent's metadata — so we never have to manually re-attach `paper_id` to thousands of individual chunks.

In [ ]:
documents = [
    Document(
        text=row['text'],
        metadata={
            'paper_id': row['paper_id'],
            'source_path': row['path']
        }
    )
    for _, row in docs_df.iterrows()
]

print(f"Created {len(documents)} Document objects\n")
print(f"Example Document:")
print(f"  Type        : {type(documents[0]).__name__}")
print(f"  paper_id    : {documents[0].metadata['paper_id']}")
print(f"  text length : {len(documents[0].text):,} chars")
print(f"  metadata    : {documents[0].metadata}")

## 5. Chunking with SentenceSplitter

**Input →** `documents`: `List[Document]` — one document per paper, full Markdown text  
**Method →** `SentenceSplitter(chunk_size=800, chunk_overlap=100)` — splits text into overlapping windows of ~800 tokens, always cutting at sentence boundaries rather than mid-sentence  
**Output →** `chunks_df` — a DataFrame with one row per chunk; each row has a stable `chunk_id`, a `paper_id` inherited from the parent document, and the chunk `text`

> **Why `SentenceSplitter` rather than a fixed-size or character-based splitter?**  
> A fixed-size splitter (e.g. `RecursiveCharacterTextSplitter` from LangChain) cuts every N characters regardless of sentence boundaries, which can split a sentence mid-clause and leave both fragments uninterpretable in isolation. `SentenceSplitter` first tokenises the text into sentences, then packs them into windows — so every chunk boundary falls between complete sentences. For scientific text, where a single sentence can carry a full quantitative claim, this distinction matters.

> **Chunk size trade-offs:**
> | Setting | Effect |
> |---|---|
> | Too large (> 1500 tokens) | Chunks contain multiple unrelated ideas; retrieval returns noisy context |
> | Too small (< 200 tokens) | Chunks lose local context; a single sentence is often uninterpretable alone |
> | 800 tokens | Comfortably fits one methods paragraph or results subsection |
> | 100-token overlap | Ensures sentences spanning a chunk boundary appear in both adjacent chunks — nothing is lost at the seam |

> **What to notice:** The example chunk printed below shows real chunk content with its metadata. Check that the text is coherent and not cut mid-sentence.


In [ ]:
splitter = SentenceSplitter(chunk_size=800, chunk_overlap=100)
nodes = splitter.get_nodes_from_documents(documents)

# Use a stable chunk_id: paper_id + zero-padded index, not a global integer
chunks_df = pd.DataFrame([
    {
        'chunk_id': f"{node.metadata.get('paper_id')}_chunk{i:04d}",
        'paper_id': node.metadata.get('paper_id'),
        'text': node.text
    }
    for i, node in enumerate(nodes)
])

print(f"Total chunks   : {len(chunks_df)}")
print(f"Total documents: {len(documents)}")
print(f"Mean chunks/doc: {len(chunks_df) / len(documents):.1f}")
print(f"\n--- Example chunk (index 0) ---")
print(f"chunk_id : {chunks_df.iloc[0]['chunk_id']}")
print(f"paper_id : {chunks_df.iloc[0]['paper_id']}")
print(f"\n{chunks_df.iloc[0]['text']}")

## 6. Embeddings with Sentence Transformers

**Input →** `chunks_df['text']` — a list of chunk strings (one per row)  
**Method →** `SentenceTransformer('all-MiniLM-L6-v2').encode()` — maps each chunk to a dense 384-dimensional vector. Semantically similar passages end up near each other in this geometric space.  
**Output →** `chunk_embeddings`: a NumPy array of shape `(n_chunks, 384)`

> **Why `all-MiniLM-L6-v2`?**
> - Fast enough to run on CPU for a corpus of this size (seconds, not minutes)
> - Open-source, no API call required — embeddings are computed locally
> - Produces strong semantic representations for English scientific text
> - 384 dimensions is compact enough that cosine similarity over ~500 chunks takes milliseconds
>
> For larger corpora, non-English text, or higher-precision needs, `all-mpnet-base-v2` or a domain-specific biomedical model (e.g., `BioLORD`) would be better choices.
>
> **What 384 dimensions means:** Each dimension encodes a learned latent feature of the text. You don't interpret individual dimensions — what matters is the *angle* between two vectors. A small angle (high cosine similarity) means the passages express related ideas.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

_cache = Path('outputs/chunk_embeddings.npy')

if _cache.exists():
    chunk_embeddings = np.load(_cache)
    print(f"Loaded embeddings from cache ({_cache})")
    print(f"  Shape: {chunk_embeddings.shape}")
    print("Delete outputs/chunk_embeddings.npy to force re-encoding.")

else:
    chunk_embeddings = embedding_model.encode(
        chunks_df['text'].tolist(),
        show_progress_bar=True
    )
    np.save(_cache, chunk_embeddings)
    print(f"\nEmbedding matrix shape : {chunk_embeddings.shape}")
    print(f"  → {chunk_embeddings.shape[0]} chunks  ×  {chunk_embeddings.shape[1]} dimensions")
    print(f"Saved to {_cache}")

print(f"\nSample: first 8 values of chunk 0's vector:")
print(np.round(chunk_embeddings[0, :8], 4))

## 7. Prompt Engineering Step 1: Query Rewriting

**Input →** `user_question` — a natural language question as a user might type it  
**Method →** GPT-4o-mini with a **query rewriting prompt** that transforms a conversational question into a retrieval-optimised declarative statement  
**Output →** `rewritten_query` — a reformulated query designed to match the vocabulary and writing style of scientific methods/results text

### Why rewrite queries?

Embedding models are trained to match semantically similar text, but user questions and scientific passages often use different vocabulary:

| User question | How papers phrase it |
|---|---|
| "Which papers discuss gene fusion detection?" | "Chimeric transcripts were identified using RNA-seq alignment..." |
| "How do you find rare mutations?" | "Variant calling was performed on whole-exome sequencing data..." |

A **declarative, keyword-rich statement** narrows the gap between the query embedding and passage embeddings, improving retrieval precision.

### Prompt Design

The prompt instructs the model to:
1. Convert the question into a declarative statement (not a question)
2. Use domain-specific terminology that would appear in a methods or results section
3. Be concise (1–2 sentences) to avoid diluting the embedding signal

In [ ]:
user_question = "Which papers discuss gene fusion detection?"

# --- Prompt Engineering: Query Rewriting ---
# The prompt is shown here explicitly so the design choices are visible.
query_rewrite_prompt = f"""You are a scientific information retrieval assistant.

Your task is to rewrite the user's question as a retrieval query optimised for \
semantic search over scientific paper abstracts and methods sections.

Rules:
- Convert the question into a declarative statement, not a question
- Use precise scientific terminology likely to appear in the target papers
- Keep the rewrite to 1–2 sentences
- Do not invent facts — only rephrase the intent

User question: {user_question}

Retrieval query:"""

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": query_rewrite_prompt}]
)

rewritten_query = response.choices[0].message.content.strip()

print("=== Query Rewriting ===\n")
print(f"  Original  : {user_question}")
print(f"  Rewritten : {rewritten_query}")

## 8. Semantic Retrieval

**Input →** `rewritten_query` (string) + `chunk_embeddings` (NumPy array, shape `n_chunks × 384`)  
**Method →** Embed the query using the same `SentenceTransformer` model, then compute **cosine similarity** between the query vector and every chunk vector using `sklearn.metrics.pairwise.cosine_similarity` in a single matrix operation  
**Output →** `top_hits` — the top-5 chunks ranked by semantic similarity score, with full text and metadata

> **Why `sklearn.metrics.pairwise.cosine_similarity` rather than a vector database?**  
> At this corpus size (~400 chunks), a vectorised NumPy/sklearn operation over the full embedding matrix completes in milliseconds — a dedicated vector store like FAISS or ChromaDB would add a dependency and setup overhead without any speed benefit. The trade-off reverses at roughly 50,000+ chunks, where approximate nearest-neighbour indices become necessary.

> **Cosine similarity** measures the angle between two vectors, not their magnitude. A score of 1.0 means the vectors point in exactly the same direction (perfect semantic match); 0.0 means orthogonal (no relationship). In practice, scores for relevant scientific passages typically fall between 0.4–0.8.
>
> **What to notice:** Inspect the scores of the top hits. If the best match is below ~0.35, either the query is too vague or the corpus genuinely doesn't contain relevant material. Notice also how results span multiple papers — the same question often has partial answers spread across your corpus.


In [ ]:
query_embedding = embedding_model.encode([rewritten_query])
scores = cosine_similarity(query_embedding, chunk_embeddings)[0]

chunks_df['score'] = scores
top_hits = chunks_df.sort_values('score', ascending=False).head(5).reset_index(drop=True)

print(f"Score range across all {len(chunks_df)} chunks: {scores.min():.3f} – {scores.max():.3f}\n")
print(f"=== Top 5 Retrieved Passages ===\n")
for _, row in top_hits.iterrows():
    print(f"[{row['paper_id']}]  score={row['score']:.4f}  chunk={row['chunk_id']}")
    print(row['text'][:500])
    print()

## 9. Prompt Engineering Step 2: Context Assembly

**Input →** `top_hits` — top-k retrieved chunks with `paper_id`, `chunk_id`, and `text`  
**Method →** Format each passage with a visible citation label and concatenate with `---` separators  
**Output →** `context` — a single formatted string ready to be injected into the generation prompt

### Why assembly format matters

The formatting choices here directly affect the quality of the generated answer:

| Choice | Purpose |
|---|---|
| `[SOURCE: paper_id \| chunk_id]` header | Gives the model explicit citation handles to reference in the answer |
| `---` separator between passages | Prevents the model from blending text from adjacent passages |
| Full chunk text (not truncated) | The model needs enough context to assess relevance and form a coherent answer |

The assembled context is the *only* evidence the model is permitted to use — we enforce this in the next step's generation prompt.

In [ ]:
context_parts = []
for _, row in top_hits.iterrows():
    header = f"[SOURCE: {row['paper_id']} | {row['chunk_id']}]"
    context_parts.append(f"{header}\n{row['text']}")

context = "\n\n---\n\n".join(context_parts)

print("=== Assembled Context (first 3000 chars) ===\n")
print(context[:3000])
print(f"\n...\n\nTotal: {len(context_parts)} passages,  {len(context):,} characters")

## 10. Prompt Engineering Step 3: Grounded Answer Generation

**Input →** `context` (assembled passages) + `user_question`  
**Method →** GPT-4.1 with a carefully designed **grounded generation prompt** that constrains the model to the retrieved evidence  
**Output →** A cited, evidence-grounded answer in natural language

### Prompt Design Principles

Each instruction in the prompt below is a deliberate design choice:

| Instruction | Why it matters |
|---|---|
| "Answer using ONLY information present in the sources" | Prevents the model drawing on parametric memory, which may be outdated or wrong |
| "Cite the source label after each claim" | Makes every statement verifiable against the retrieved passages |
| "Synthesize across sources when multiple cover the same topic" | Avoids a naïve list; forces the model to compare and integrate evidence |
| "If the answer cannot be found, say so explicitly" | Prevents confident-sounding hallucination when the context is insufficient |

Removing any one of these instructions produces measurably worse answers: uncited answers are unverifiable, non-synthesizing answers are repetitive, and answers without the uncertainty instruction hallucinate when the corpus has gaps.

In [ ]:
# --- Prompt Engineering: Grounded Generation ---
# The full prompt is written out here so every design choice is visible.
generation_prompt = f"""You are a scientific research assistant. Your task is to answer \
a question using ONLY the source passages provided below.

Instructions:
1. Answer using ONLY the literal text of the passages below. Do not use any knowledge \
about these papers or topics from your training data — even if you recognise the paper titles.
2. Every sentence must be directly traceable to a word or phrase in one of the passages. \
If a detail is not stated in a passage, do not include it.
3. After each claim, cite the source using its label in square brackets, \
e.g. [SOURCE: paper_id | chunk_id].
4. If multiple sources discuss the same topic, synthesize them into a unified answer \
rather than listing them separately.
5. If the answer cannot be found in the provided sources, respond with exactly: \
"The retrieved evidence does not contain sufficient information to answer this question."

--- SOURCES ---
{context}
--- END SOURCES ---

Question: {user_question}

Answer:"""

response = client.chat.completions.create(
    model='gpt-4.1',
    messages=[{"role": "user", "content": generation_prompt}]
)

answer = response.choices[0].message.content.strip()
print("=== Grounded Answer ===\n")
print(answer)


## 11. Full-Corpus Triplet Extraction

**Input →** `chunks_df` — all chunks from all papers  
**Method →** GPT-4o-mini with a structured extraction prompt, run on every chunk; `response_format={"type": "json_object"}` enforces parseable output  
**Output →** `triplets_raw_df` — a raw DataFrame of all extracted SPO triplets across the entire corpus

### Why full-corpus extraction?

Extracting from **all chunks** builds a corpus-wide index: a graph that captures relationships across all your papers regardless of what you query. This is what allows graph traversal to find things that vector similarity missed — the graph was built from the whole corpus, not just the current search results.

### Cost and time note

This step makes one API call per chunk. For a 12-paper corpus (~500 chunks) expect:
- **Time:** ~40 minutes
- **Cost:** ~$0.20–0.50 (gpt-4o-mini)

This is an **offline index-building step** — in production you would run it once and persist the results. The `tqdm` progress bar makes the wait visible.

In [ ]:
from tqdm.auto import tqdm

# Canonical predicate vocabulary used at both extraction and normalisation.
# Keeping this list here means both cells stay in sync — edit once, applies everywhere.
CANONICAL_PREDICATES = [
    "detects",
    "identifies",
    "uses",
    "causes",
    "treats",
    "encodes",
    "regulates",
    "inhibits",
    "activates",
    "is associated with",
    "is a type of",
    "is part of",
    "is expressed in",
    "interacts with",
    "evaluates",
]

_cache = Path('outputs/triplets_raw.csv')

if _cache.exists():
    triplets_raw_df = pd.read_csv(_cache)
    print(f"Loaded {len(triplets_raw_df)} triplets from cache ({_cache})")
    print("Delete outputs/triplets_raw.csv to force re-extraction.")

else:
    canonical_pred_list = "\n".join(f'  "{p}"' for p in CANONICAL_PREDICATES)

    triplet_prompt_template = """Extract subject-predicate-object triplets from the scientific passage below.

Return a JSON object with a single key "triplets" whose value is an array of objects.
Each object must have exactly these keys:
  "subject"   — a concise noun phrase (1–5 words) naming a specific biological entity,
                tool, method, or disease. Use standard scientific terminology.
                GOOD: "RNA-seq", "STAR-Fusion", "gene fusion", "rare disease", "diagnostic yield"
                BAD:  "fusion events in rare disease cohort with limited success"
                BAD:  "diagnosis of up to 40% of genetic disease cases"
                BAD:  "any RNA-Seq analysis aimed at diagnosis of rare disease"
                NOT:  "study", "paper", "authors", "figure", "table", "section"
  "predicate" — choose the closest match from this vocabulary:
{canonical_predicates}
                If none fits precisely, use the shortest accurate verb phrase (2–4 words).
  "object"    — a concise noun phrase (1–5 words) naming a specific biological entity,
                tool, method, or disease. Same rules as subject.
                GOOD: "fusion transcripts", "RNA-seq", "hereditary disease", "sensitivity"
                BAD:  "improvements in throughput, sensitivity and specificity over traditional approaches"
                BAD:  "figure N", "table N", "et al.", section references, citation numbers
  "source"    — copy the source label exactly as provided

Extract 3–6 triplets per passage. Focus on: genes, proteins, diseases, algorithms, methods,
organisms, and their relationships. Do NOT extract triplets about document structure,
comparisons, or statistics.

Source label: {{source_label}}
Passage:
{{passage}}"""

    # Insert the canonical predicate list into the template
    triplet_prompt_template = triplet_prompt_template.format(
        canonical_predicates=canonical_pred_list
    )

    all_triplets = []
    failed_chunks = []

    for _, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Extracting triplets"):
        prompt = triplet_prompt_template.format(
            source_label=row['chunk_id'],
            passage=row['text']
        )
        try:
            response = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )
            raw = response.choices[0].message.content.strip()
            parsed = json.loads(raw)
            triplets = parsed.get("triplets", parsed) if isinstance(parsed, dict) else parsed
            for t in triplets:
                t['paper_id'] = row['paper_id']
                t['chunk_id'] = row['chunk_id']
            all_triplets.extend(triplets)
        except Exception as e:
            failed_chunks.append(row['chunk_id'])

    triplets_raw_df = pd.DataFrame(all_triplets)
    triplets_raw_df.to_csv(_cache, index=False)

    print(f"Chunks processed   : {len(chunks_df)}")
    print(f"Raw triplets       : {len(triplets_raw_df)}")
    if failed_chunks:
        print(f"Failed chunks      : {len(failed_chunks)}")
    print(f"Saved to {_cache}")

print(f"\nSample of raw triplets:")
triplets_raw_df[['subject', 'predicate', 'object', 'paper_id']].head(10)


## 12. Junk Filter

**Input →** `triplets_raw_df` — all raw extracted triplets including noise  
**Method →** Rule-based filtering on subject, predicate, and object fields; a simple `apply` over the DataFrame — no LLM call needed  
**Output →** `triplets_filtered` — triplets with document-structure references, citation artefacts, and trivial predicates removed

### What gets filtered

| Junk type | Example | Why |
|---|---|---|
| Document structure | "result — is shown in — Figure 4" | "Figure 4" is not a knowledge entity |
| Citation artefact | "method — was described by — Smith et al." | Author names are not domain concepts |
| Numeric object | "value — equals — 0.05" | Numbers without context are meaningless graph nodes |
| Weak predicate | "RNA-seq — is — a technique" | "is a" adds no relational knowledge |
| Too short | "it — does — x" | Likely a parsing artefact |

### Why rule-based rather than another LLM call?

These patterns are deterministic and cheap to check. Using an LLM to filter junk would be slower and more expensive without any gain in accuracy for these specific cases. LLM calls are reserved for tasks that require semantic understanding — like normalisation in the next step.

In [ ]:
JUNK_PREFIXES = {
    'figure', 'fig.', 'fig ', 'table', 'supplementary', 'supp',
    'appendix', 'section', 'ref.', 'reference', '[', '('
}

WEAK_PREDICATES = {
    'is', 'is a', 'is an', 'are', 'was', 'were',
    'has', 'have', 'had', 'be', 'been'
}

def is_junk(row):
    subj = str(row.get('subject', '')).lower().strip()
    obj  = str(row.get('object', '')).lower().strip()
    pred = str(row.get('predicate', '')).lower().strip()

    # Document structure references
    if any(subj.startswith(p) for p in JUNK_PREFIXES):
        return True
    if any(obj.startswith(p) for p in JUNK_PREFIXES):
        return True
    # Citation artefacts
    if 'et al' in subj or 'et al' in obj:
        return True
    # Numeric-only entities (page numbers, figure numbers)
    if obj.replace('.', '').replace(' ', '').isdigit():
        return True
    # Weak/trivial predicates
    if pred in WEAK_PREDICATES:
        return True
    # Too short to be meaningful
    if len(subj) < 3 or len(obj) < 3:
        return True
    return False

junk_mask = triplets_raw_df.apply(is_junk, axis=1)
triplets_filtered = triplets_raw_df[~junk_mask].reset_index(drop=True)

print(f"Raw triplets       : {len(triplets_raw_df)}")
print(f"Junk removed       : {junk_mask.sum()}")
print(f"Filtered triplets  : {len(triplets_filtered)}")
print(f"\nSample of removed triplets:")
triplets_raw_df[junk_mask][['subject', 'predicate', 'object']].head(10)

## 13. Entity and Predicate Normalisation

**Input →** `triplets_filtered` — junk-free triplets with noisy entity and predicate surface forms  
**Method →** Two GPT-4o-mini calls: one to canonicalise entity names, one to normalise predicate phrases; results applied as a lookup mapping across the DataFrame  
**Output →** `triplets_clean` — triplets with consistent entity names and predicate vocabulary; `entity_map` and `predicate_map` for inspection

### Why two separate calls?

Entities and predicates have different normalisation semantics:
- **Entities** need grouping into real-world referents: "STAR aligner" and "STAR algorithm" are the same tool
- **Predicates** need synonym collapsing: "is used to detect" and "enables detection of" express the same relationship

Doing them separately gives the model a focused task and produces cleaner mappings.

### What to notice

The output shows how many variants were collapsed. A high number of entity changes indicates your corpus uses inconsistent nomenclature across papers — exactly the kind of noise that would fragment a graph without this step. After normalisation, the same entity will have one node in the graph regardless of how different papers refer to it.

In [ ]:
_cache = Path('outputs/triplets_clean.csv')

if _cache.exists():
    triplets_clean = pd.read_csv(_cache)
    print(f"Loaded {len(triplets_clean)} normalised triplets from cache ({_cache})")
    print("Delete outputs/triplets_clean.csv to force re-normalisation.")

else:
    all_entities = sorted(set(
        triplets_filtered['subject'].tolist() +
        triplets_filtered['object'].tolist()
    ))
    all_predicates = sorted(triplets_filtered['predicate'].unique().tolist())

    print(f"Unique entities before normalisation  : {len(all_entities)}")
    print(f"Unique predicates before normalisation: {len(all_predicates)}")

    # --- Entity Normalisation (batched to avoid output-token truncation) ---
    ENTITY_BATCH_SIZE = 200  # ~200 entities keeps response well under gpt-4o-mini's 4096-token output limit

    def _normalise_entity_batch(batch: list[str]) -> dict:
        prompt = f"""You are normalising entity names extracted from scientific papers about
genomics, bioinformatics, rare diseases, and precision oncology.

Many names in the list below refer to the same real-world entity with different surface forms.
Examples:
  "STAR" / "STAR aligner" / "STAR algorithm"     → canonical: "STAR"
  "RNA-seq" / "RNA sequencing" / "bulk RNA-seq"   → canonical: "RNA-seq"
  "BRCA2" / "BRCA-2" / "breast cancer gene 2"     → canonical: "BRCA2"

Return a JSON object where keys are the original entity names (exactly as provided)
and values are the canonical form to use.

Rules:
- Use the most precise, widely-used scientific name as the canonical form
- Prefer abbreviations over long-form when both are standard (e.g. "RNA-seq" not "ribonucleic acid sequencing")
- If a name is already canonical, map it to itself
- Do not invent new names — only choose among the variants provided

Entity list:
{json.dumps(batch, indent=2)}"""

        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)

    entity_map: dict = {}
    batches = [all_entities[i:i + ENTITY_BATCH_SIZE] for i in range(0, len(all_entities), ENTITY_BATCH_SIZE)]
    print(f"\nNormalising entities in {len(batches)} batches of up to {ENTITY_BATCH_SIZE}...")
    for idx, batch in enumerate(batches, 1):
        print(f"  Batch {idx}/{len(batches)} ({len(batch)} entities)...", end=" ", flush=True)
        batch_map = _normalise_entity_batch(batch)
        entity_map.update(batch_map)
        print("done")

    entity_changes = {k: v for k, v in entity_map.items() if k != v}
    print(f"\nEntity variants normalised: {len(entity_changes)}")
    for orig, canon in list(entity_changes.items())[:10]:
        print(f"  '{orig}'  →  '{canon}'")

    # --- Predicate Normalisation ---
    # Map every predicate to the nearest entry in CANONICAL_PREDICATES (defined in the
    # extraction cell). Using a fixed target vocabulary guarantees a small, consistent
    # predicate set rather than letting the LLM invent its own canonical forms.
    canonical_pred_list = "\n".join(f'  "{p}"' for p in CANONICAL_PREDICATES)

    pred_norm_prompt = f"""Normalise these predicate phrases from scientific papers by mapping each one
to the single closest entry in the canonical vocabulary below.

Canonical vocabulary (use ONLY these values as output):
{canonical_pred_list}

Rules:
- Every predicate must map to exactly one entry from the canonical vocabulary above.
- Choose the closest semantic match.
- If genuinely ambiguous, prefer the more general option.

Return a JSON object mapping each original predicate (key) to its canonical form (value).

Predicates to normalise:
{json.dumps(all_predicates, indent=2)}"""

    pred_response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": pred_norm_prompt}],
        response_format={"type": "json_object"}
    )
    predicate_map = json.loads(pred_response.choices[0].message.content)

    pred_changes = {k: v for k, v in predicate_map.items() if k != v}
    print(f"\nPredicate variants normalised: {len(pred_changes)}")
    for orig, canon in pred_changes.items():
        print(f"  '{orig}'  →  '{canon}'")

    # --- Apply both mappings ---
    triplets_clean = triplets_filtered.copy()
    triplets_clean['subject']   = triplets_clean['subject'].map(lambda x: entity_map.get(x, x))
    triplets_clean['object']    = triplets_clean['object'].map(lambda x: entity_map.get(x, x))
    triplets_clean['predicate'] = triplets_clean['predicate'].map(lambda x: predicate_map.get(x, x))

    triplets_clean.to_csv(_cache, index=False)
    print(f"\nSaved to {_cache}")

unique_entities_after = len(set(
    triplets_clean['subject'].tolist() + triplets_clean['object'].tolist()
))
print(f"\nUnique entities  : {unique_entities_after}")
print(f"Unique predicates: {len(triplets_clean['predicate'].unique())}")
triplets_clean[['subject', 'predicate', 'object', 'paper_id']].head(10)

## 14. Building the Knowledge Graph

**Input →** `triplets_clean` — normalised, junk-free SPO triplets from the entire corpus  
**Method →** `networkx.MultiDiGraph` — each triplet maps directly to a directed edge; node size in the visualisation is scaled by degree  
**Output →** `G`: a full-corpus NetworkX directed graph; `degree_series`: entities ranked by connectivity; a matplotlib visualisation

> **Why NetworkX rather than an RDF triplestore?**  
> The data here is already in SPO form, which maps directly onto RDF triples — so `rdflib` or a triplestore like Apache Jena would be a technically valid alternative. The reason to prefer NetworkX at this stage is that our queries are algorithmic: degree centrality, neighbour lookup, shortest-path traversal. These are native operations in NetworkX and require only Python. An RDF triplestore earns its complexity when you need SPARQL (a full declarative query language) or want to federate your extracted graph against external biomedical ontologies such as HPO or OMIM — for example, to ask "find all entities that are a subclass of rare disease as defined by ORDO." That capability is out of scope here but is a natural next step if the pipeline grows. See the Next Steps section at the end of the notebook.  
> We use `MultiDiGraph` rather than `DiGraph` so that multiple distinct relationships between the same pair of entities (e.g., `RNA-seq` both `detects` and `evaluates` `gene fusions`) are preserved as separate edges rather than the later one silently overwriting the earlier.  
> The main alternative within the Python graph ecosystem is `igraph`, which is faster for very large graphs but requires a compiled C extension and has a less Pythonic API. At scales where NetworkX becomes slow (millions of nodes/edges), the right move is to export to a purpose-built graph database like Neo4j.

### From triplets to a graph

| Triplet field | Graph element |
|---|---|
| `subject` | Source node |
| `predicate` | Directed edge label |
| `object` | Target node |
| `chunk_id` | Edge attribute (provenance — traceable back to a specific passage) |

Because the graph is built from the **entire corpus**, highly connected nodes represent concepts that appear across multiple papers. This is a meaningful signal — if "RNA-seq" connects to 40 other entities, it is genuinely central to your research area, not just prominent in one paper.

> **What to notice:** Compare the top nodes by degree to your own sense of which concepts are central to your corpus. If the graph is capturing domain knowledge correctly, the most connected entities should be the tools, diseases, and methods you use most across your work. The isolated-node percentage (degree ≤ 1) is a useful quality signal — a high percentage suggests entity names are too specific to appear across multiple chunks.


In [ ]:
_cache = Path('outputs/knowledge_graph.graphml')

if _cache.exists():
    G = nx.read_graphml(_cache)
    degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False)
    print(f"Loaded graph from cache ({_cache})")
    print(f"  Nodes : {G.number_of_nodes()}")
    print(f"  Edges : {G.number_of_edges()}")
    isolated = sum(1 for n in G.nodes() if G.degree(n) <= 1)
    print(f"  Nodes with degree ≤ 1 (leaf/isolated): {isolated} ({isolated/G.number_of_nodes()*100:.1f}%)")
    print(f"\nTop 20 entities by degree:")
    print(degree_series.head(20).to_string())
    print("\nDelete outputs/knowledge_graph.graphml to force rebuild.")

else:
    # MultiDiGraph allows multiple edges between the same node pair (different predicates).
    # DiGraph would silently overwrite earlier edges when subject+object repeat across chunks.
    G = nx.MultiDiGraph()

    for _, row in triplets_clean.iterrows():
        subj = str(row.get('subject', '')).strip()
        pred = str(row.get('predicate', '')).strip()
        obj  = str(row.get('object', '')).strip()
        src  = str(row.get('chunk_id', row.get('paper_id', ''))).strip()
        if subj and pred and obj:
            G.add_edge(subj, obj, predicate=pred, source=src)

    # degree_series reused in §15 and §16
    degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False)
    isolated = sum(1 for n in G.nodes() if G.degree(n) <= 1)

    print("=== Full-Corpus Knowledge Graph ===")
    print(f"  Nodes (unique entities) : {G.number_of_nodes()}")
    print(f"  Edges (relationships)   : {G.number_of_edges()}")
    print(f"  Nodes with degree ≤ 1 (leaf/isolated): {isolated} ({isolated/G.number_of_nodes()*100:.1f}%)")
    print(f"\nTop 20 entities by degree:")
    print(degree_series.head(20).to_string())

    # Save immediately so subsequent runs load from cache
    nx.write_graphml(G, _cache)
    print(f"\nSaved to {_cache}")

    # --- Visualisation ---
    fig, ax = plt.subplots(figsize=(16, 12))
    pos = nx.spring_layout(G, seed=42, k=2.0)

    node_sizes = [300 + 150 * G.degree(n) for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='steelblue', alpha=0.85, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_color='white', font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='#888888', arrows=True,
                           arrowsize=12, connectionstyle='arc3,rad=0.08', ax=ax, alpha=0.6)
    edge_labels = {(u, v, k): d['predicate'] for u, v, k, d in G.edges(data=True, keys=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels={(u,v): d for (u,v,k), d in edge_labels.items()},
                                 font_size=6, label_pos=0.4, ax=ax)

    ax.set_title("Full-Corpus Knowledge Graph (normalised entities and predicates)", fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('outputs/knowledge_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Visualisation saved to outputs/knowledge_graph.png")


## 15. Exploring the Knowledge Graph

A graph is only as useful as what you can ask of it. Two immediately practical operations:

**Neighborhood query** — given an entity, what does the full corpus say about it?  
This is an instantaneous graph lookup, not another embedding search or LLM call.

**Path query** — is there a chain of relationships connecting entity A to entity B?  
Useful for discovering indirect connections across papers that keyword or vector search would miss entirely.

> **Note on the next step:** These traversal operations are exactly what §16 uses to expand retrieval. Understanding them here makes the graph-augmented retrieval step intuitive — it's just neighbourhood lookup applied systematically to the entities found in the vector results.

In [ ]:
# degree_series was computed in §14 — reused here
top_node = degree_series.index[0]

print(f"=== Relationships involving '{top_node}' ===\n")
print("  Outgoing (this entity as subject):")
for _, nbr, data in G.out_edges(top_node, data=True):
    pred = data.get('predicate', '?')
    src  = data.get('source', '?')
    print(f"    {top_node}  --[{pred}]-->  {nbr}  ({src})")

print("\n  Incoming (this entity as object):")
for pre, _, data in G.in_edges(top_node, data=True):
    pred = data.get('predicate', '?')
    src  = data.get('source', '?')
    print(f"    {pre}  --[{pred}]-->  {top_node}  ({src})")

if len(degree_series) >= 2:
    node_a, node_b = degree_series.index[0], degree_series.index[1]
    print(f"\n=== Shortest path: '{node_a}'  →  '{node_b}' ===\n")
    try:
        path = nx.shortest_path(G, source=node_a, target=node_b)
        print(f"  Directed path  : {' → '.join(path)}")
    except nx.NetworkXNoPath:
        try:
            path = nx.shortest_path(G.to_undirected(), source=node_a, target=node_b)
            print(f"  Undirected path: {' → '.join(path)}")
        except nx.NetworkXNoPath:
            print("  No path found between these nodes.")

## 16. Graph-Augmented Retrieval

**Input →** `top_hits` (vector results) + `G` (full-corpus graph) + `triplets_clean` + `chunks_df`  
**Method →** Entity expansion via graph traversal: extract entities from retrieved chunks, find their graph neighbours, retrieve additional chunks mentioning those neighbours  
**Output →** `graph_hits` — new chunks found via graph expansion; `expansion_entities` — the neighbour concepts that bridged the gap

### The GraphRAG loop

```
top_hits
  → extract seed entities  (from triplets_clean, no LLM call needed)
  → look up in G           → find direct graph neighbours
  → subtract seed entities → expansion_entities (what the graph added)
  → search chunks_df       → passages mentioning expansion entities
  → deduplicate            → graph_hits (top 3 new chunks)
```

> **Why this finds things vector search misses:** Vector similarity finds passages *phrased like the query*. Graph expansion finds passages about concepts that your corpus says are *related to the query entities* — regardless of surface phrasing. A passage about a method that "detects" the same targets as your query entity may score poorly on cosine similarity but be highly relevant.

> **What to notice:** The `expansion_via` field shows exactly which graph relationship bridged the gap between the original query and the new chunk. This makes the reasoning transparent — you can trace every retrieved passage back to a specific graph edge.

In [ ]:
# Step 1: identify entities present in the vector-retrieved chunks
top_hit_chunk_ids = set(top_hits['chunk_id'])
seed_triplets = triplets_clean[triplets_clean['chunk_id'].isin(top_hit_chunk_ids)]
seed_entities = set(seed_triplets['subject'].tolist() + seed_triplets['object'].tolist())

print(f"Seed entities from top_hits ({len(seed_entities)}):")
for e in sorted(seed_entities):
    print(f"  {e}")

# Step 2: find their direct neighbours in the full-corpus graph
neighbour_entities = set()
for entity in seed_entities:
    if entity in G:
        neighbour_entities.update(G.neighbors(entity))
        neighbour_entities.update(G.predecessors(entity))

expansion_entities = neighbour_entities - seed_entities
print(f"\nGraph neighbours (expansion candidates): {len(expansion_entities)}")
for e in sorted(expansion_entities):
    print(f"  {e}")

# Step 3: retrieve chunks mentioning expansion entities (not already in top_hits).
# regex=False treats each entity as a literal string, avoiding misinterpretation of
# special characters (parentheses, hyphens, semicolons) that appear in entity names.
MAX_GRAPH_HITS = 10  # increase from 3 to surface more graph-expanded evidence

expansion_rows = []
for entity in expansion_entities:
    mask = (
        chunks_df['text'].str.contains(entity, case=False, na=False, regex=False) &
        ~chunks_df['chunk_id'].isin(top_hit_chunk_ids)
    )
    matches = chunks_df[mask].copy()
    matches['expansion_via'] = entity
    expansion_rows.append(matches)

if expansion_rows:
    all_graph_hits = (
        pd.concat(expansion_rows)
        .drop_duplicates(subset=['chunk_id'])
        .reset_index(drop=True)
    )
    print(f"\nTotal unique graph-expanded chunks before cap: {len(all_graph_hits)}")
    graph_hits = all_graph_hits.head(MAX_GRAPH_HITS)
    print(f"Graph-expanded chunks used (cap={MAX_GRAPH_HITS}): {len(graph_hits)}")
    for _, row in graph_hits.iterrows():
        print(f"\n  [{row['paper_id']}]  via: '{row['expansion_via']}'")
        print(f"  {row['text'][:300]}")
else:
    graph_hits = pd.DataFrame(columns=chunks_df.columns.tolist() + ['expansion_via'])
    print("\nNo additional chunks found via graph expansion for this query.")


## 17. Vector-Only vs Graph-Augmented Answer

**Input →** `context` (vector-only) + `enriched_context` (vector + graph expansion)  
**Method →** The same grounded generation prompt from §10, applied to both contexts  
**Output →** Two answers printed side by side; a summary of what the graph expansion added

### The payoff

This is the moment the notebook has been building toward. The comparison makes concrete:

- What the pure vector search found
- What graph traversal added — which new entities and chunks it surfaced
- Whether the final answer is materially richer

**Interpreting the result honestly:** If both answers are very similar, that is not a failure. It means either (a) vector retrieval was already comprehensive for this query, or (b) the graph expansion found adjacent chunks that didn't add new information. Both are valid outcomes worth understanding. The graph augmentation is most valuable for queries that touch concepts *related to but not identical to* the most prominent topics in your corpus — where vector similarity alone undershoots.

In [ ]:
# Assemble enriched context: original vector hits + graph-expanded chunks
if not graph_hits.empty:
    graph_context_parts = []
    for _, row in graph_hits.iterrows():
        header = (
            f"[SOURCE: {row['paper_id']} | {row['chunk_id']} "
            f"| graph-expanded via: '{row['expansion_via']}']"
        )
        graph_context_parts.append(f"{header}\n{row['text']}")
    enriched_context = context + "\n\n---\n\n" + "\n\n---\n\n".join(graph_context_parts)
else:
    enriched_context = context

enriched_prompt = f"""You are a scientific research assistant. Answer the question using ONLY the source passages below.

Instructions:
1. Answer using ONLY the literal text of the passages below. Do not use any knowledge \
about these papers or topics from your training data — even if you recognise the paper titles.
2. Every sentence must be directly traceable to a word or phrase in one of the passages. \
If a detail is not stated in a passage, do not include it.
3. Cite each source after claims using its label, e.g. [SOURCE: paper_id | chunk_id].
4. Synthesize across sources when multiple cover the same topic.
5. If the answer cannot be found in the sources, say so explicitly.

--- SOURCES ---
{enriched_context}
--- END SOURCES ---

Question: {user_question}

Answer:"""

enriched_response = client.chat.completions.create(
    model='gpt-4.1',
    messages=[{"role": "user", "content": enriched_prompt}]
)
graph_answer = enriched_response.choices[0].message.content.strip()

# --- Side-by-side comparison ---
divider = "=" * 70
print(divider)
print(f"VECTOR-ONLY ANSWER  ({len(top_hits)} chunks, cosine similarity)")
print(divider)
print(answer)

print(f"\n{divider}")
n_graph = len(graph_hits)
expansion_entities = list(graph_hits['expansion_via'].unique()) if not graph_hits.empty else []
print(f"GRAPH-AUGMENTED ANSWER  ({len(top_hits)} vector chunks + {n_graph} graph-expanded chunks)")
if expansion_entities:
    print(f"  Expanded via entities: {expansion_entities}")
print(divider)
print(graph_answer)

print(f"\n{divider}")
if n_graph > 0:
    print(f"Graph expansion added {n_graph} new chunk(s) not in the original top-5.")
    print("Check whether the graph-augmented answer contains information absent from the vector-only answer.")
else:
    print("Graph expansion found no additional chunks for this query.")
    print("This can mean: (a) the vector retrieval already found the relevant passages,")
    print("or (b) the query entities have sparse connections in the graph.")


## 18. Evaluation

**Input →** `answer` (vector-only), `graph_answer` (graph-augmented), `context`, `enriched_context`  
**Method →** Three complementary evaluations: one programmatic, two LLM-as-judge  
**Output →** Citation validity report, faithfulness scores with per-claim breakdown, answer delta summary

### Three evaluations, three different questions

| Evaluation | Question it answers | Method |
|---|---|---|
| **Citation validity** | Did the model cite sources that were actually in the context? | Programmatic regex check — no LLM needed |
| **Faithfulness** | Is each claim in the answer supported by the retrieved passages? | LLM-as-judge |
| **Answer delta** | What did graph expansion add that vector search alone missed? | LLM-as-judge |

### Why these three specifically?

**Citation validity** catches the most common silent failure in RAG: a model that produces well-formatted citations to chunk IDs that don't exist. This is pure hallucination dressed up as attribution. It's detectable without any ground truth.

**Faithfulness** measures the core RAG guarantee — that the answer is grounded in retrieved evidence, not the model's parametric memory. A score of 1.0 means every claim traces back to a retrieved passage. An unsupported claim is a hallucination, regardless of how plausible it sounds.

**Answer delta** closes the loop on the graph augmentation experiment. Without it, the §17 comparison relies on visual inspection. The delta gives a structured, auditable account of what — if anything — the graph contributed.

### Limitations

These evaluations use an LLM as the judge, which introduces its own biases and inconsistencies. Faithfulness scores in particular can vary between runs. For a portfolio artifact this is acceptable; for a production system you would want human-labelled ground truth and metrics like RAGAS or TruLens running over many queries.

In [ ]:
import re

def extract_cited_chunk_ids(text):
    """Extract chunk_id substrings from [SOURCE: ...] citations in an answer."""
    raw = re.findall(r'\[SOURCE:\s*([^\]]+)\]', text)
    return [r.strip() for r in raw]

# Build the set of chunk_ids that are legitimately in context
valid_chunk_ids = set(top_hits['chunk_id'])
if not graph_hits.empty:
    valid_chunk_ids.update(graph_hits['chunk_id'])

def check_citations(answer_text, label):
    cited = extract_cited_chunk_ids(answer_text)
    if not cited:
        print(f"{label}: no citations found in answer.")
        return
    valid   = [c for c in cited if any(vid in c for vid in valid_chunk_ids)]
    invalid = [c for c in cited if not any(vid in c for vid in valid_chunk_ids)]
    print(f"{label}:")
    print(f"  Citations found       : {len(cited)}")
    print(f"  Valid (in context)    : {len(valid)}")
    print(f"  Invalid (hallucinated): {len(invalid)}")
    for c in invalid:
        print(f"    ⚠  [{c}]")

print("=== Citation Validity ===\n")
check_citations(answer, "Vector-only answer")
print()
check_citations(graph_answer, "Graph-augmented answer")

In [ ]:
# --- Faithfulness: vector-only answer ---
faithfulness_prompt_template = """You are evaluating a RAG system's answer for faithfulness to its retrieved sources.

RETRIEVED CONTEXT:
{context}

ANSWER TO EVALUATE:
{answer}

Task:
1. Break the answer into individual factual claims. Ignore hedging phrases like "according to the sources".
2. For each claim, judge whether it is SUPPORTED or UNSUPPORTED by the context above.
   A claim is supported only if the context contains information that directly backs it up.
3. Return a JSON object with:
   - "claims": array of objects, each with "claim" (string) and "supported" (boolean)
   - "faithfulness_score": supported_claims / total_claims as a float (0.0–1.0)
   - "summary": one sentence describing the overall faithfulness

Return only the JSON object."""

def evaluate_faithfulness(answer_text, context_text):
    # Pass the full context so no supporting evidence is truncated.
    # gpt-4o-mini has a 1M token context window; the enriched context is well within limits.
    prompt = faithfulness_prompt_template.format(
        context=context_text,
        answer=answer_text
    )
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

vector_faith  = evaluate_faithfulness(answer, context)
graph_faith   = evaluate_faithfulness(graph_answer, enriched_context)

print("=== Faithfulness Scores ===\n")
print(f"Vector-only answer   : {vector_faith['faithfulness_score']:.2f}  — {vector_faith['summary']}")
print(f"Graph-augmented answer: {graph_faith['faithfulness_score']:.2f}  — {graph_faith['summary']}")

print(f"\nVector-only claim breakdown:")
for c in vector_faith['claims']:
    mark = "✓" if c['supported'] else "✗"
    print(f"  {mark}  {c['claim']}")

print(f"\nGraph-augmented claim breakdown:")
for c in graph_faith['claims']:
    mark = "✓" if c['supported'] else "✗"
    print(f"  {mark}  {c['claim']}")

# --- Answer Delta: what did graph expansion add? ---
delta_prompt = f"""Compare these two answers to the same question.

VECTOR-ONLY ANSWER:
{answer}

GRAPH-AUGMENTED ANSWER:
{graph_answer}

Task: Identify factual claims present in the GRAPH-AUGMENTED answer that are absent from the VECTOR-ONLY answer.
These represent the information added by graph expansion.

Return a JSON object with:
- "new_claims": array of strings (each a claim unique to the graph-augmented answer)
- "summary": one sentence describing what the graph expansion contributed,
  or "Graph expansion did not add materially new information." if the answers are equivalent

Return only the JSON object."""

delta_response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": delta_prompt}],
    response_format={"type": "json_object"}
)
delta = json.loads(delta_response.choices[0].message.content)

print(f"\n=== Answer Delta (what graph expansion added) ===\n")
print(f"Summary: {delta['summary']}")
if delta['new_claims']:
    print(f"\nNew claims in graph-augmented answer ({len(delta['new_claims'])}):")
    for claim in delta['new_claims']:
        print(f"  + {claim}")
else:
    print("No new claims identified.")


## 19. Save Outputs

We write all pipeline artefacts to `outputs/` for reproducibility and downstream use.

| File | Contents |
|---|---|
| `chunks.csv` | All chunks with `paper_id` and `text` |
| `top_hits.csv` | Vector-retrieved passages with similarity scores |
| `triplets_raw.csv` | All raw extracted triplets before filtering (cached by §11) |
| `triplets_clean.csv` | Filtered and normalised triplets used to build the graph (cached by §13) |
| `knowledge_graph.graphml` | Full graph — loadable by Gephi, Cytoscape, or NetworkX (cached by §14) |
| `knowledge_graph.png` | Static visualisation of the graph |
| `graph_hits.csv` | Graph-expanded chunks for the demo query |
| `evaluation.json` | Citation validity, faithfulness scores, and answer delta |

In [ ]:
chunks_df.drop(columns=['score'], errors='ignore').to_csv('outputs/chunks.csv', index=False)
top_hits.to_csv('outputs/top_hits.csv', index=False)
if not graph_hits.empty:
    graph_hits.to_csv('outputs/graph_hits.csv', index=False)

# triplets_raw, triplets_clean, and knowledge_graph.graphml are saved by their
# respective cells the first time they are computed; no need to re-save here.

# Save evaluation results
evaluation_results = {
    'query': user_question,
    'vector_only': {
        'faithfulness_score': vector_faith['faithfulness_score'],
        'faithfulness_summary': vector_faith['summary'],
        'claims': vector_faith['claims'],
    },
    'graph_augmented': {
        'faithfulness_score': graph_faith['faithfulness_score'],
        'faithfulness_summary': graph_faith['summary'],
        'claims': graph_faith['claims'],
        'new_claims_vs_vector': delta['new_claims'],
        'delta_summary': delta['summary'],
    }
}
with open('outputs/evaluation.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print("Saved to outputs/:")
for f in sorted(Path('outputs').iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size_kb:>8.1f} KB")

---

## 20. Just Ask Questions

After twelve sections of pipeline construction, evaluation metrics, and comparison tables, it is worth pausing to remember what this was all for: being able to ask questions about your own research literature and get grounded, cited answers back in seconds.

The cells below ask a handful of questions relevant to this corpus — no side-by-side comparisons, no faithfulness scoring, no infrastructure discussion. Just the pipeline doing what it was built to do.


In [ ]:
def ask(question, top_k=5):
    """Run the full graph-augmented RAG pipeline on a single question and print the answer."""

    # Step 1: rewrite the query
    rw = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": (
            "Rewrite the following question as a declarative statement using "
            "precise scientific terminology suitable for searching methods and results sections "
            "of genomics and rare disease research papers. Return only the rewritten statement.\n\n"
            f"Question: {question}"
        )}]
    )
    rewritten = rw.choices[0].message.content.strip()

    # Step 2: vector retrieval
    q_vec = embedding_model.encode([rewritten])
    sims = cosine_similarity(q_vec, chunk_embeddings)[0]
    top_idx = sims.argsort()[::-1][:top_k]
    hits = chunks_df.iloc[top_idx].copy()
    hits['score'] = sims[top_idx]
    hit_ids = set(hits['chunk_id'])

    # Step 3: graph expansion
    seed_t = triplets_clean[triplets_clean['chunk_id'].isin(hit_ids)]
    seeds  = set(seed_t['subject'].tolist() + seed_t['object'].tolist())
    neighbours = set()
    for e in seeds:
        if e in G:
            neighbours.update(G.neighbors(e))
            neighbours.update(G.predecessors(e))
    expansion = neighbours - seeds

    exp_rows = []
    for e in expansion:
        mask = (
            chunks_df['text'].str.contains(e, case=False, na=False, regex=False) &
            ~chunks_df['chunk_id'].isin(hit_ids)
        )
        m = chunks_df[mask].copy()
        m['expansion_via'] = e
        exp_rows.append(m)

    if exp_rows:
        g_hits = pd.concat(exp_rows).drop_duplicates(subset=['chunk_id']).head(3)
        graph_parts = []
        for _, row in g_hits.iterrows():
            header = f"[SOURCE: {row['paper_id']} | {row['chunk_id']} | graph-expanded via: '{row['expansion_via']}']"
            graph_parts.append(f"{header}\n{row['text']}")
        extra = "\n\n---\n\n" + "\n\n---\n\n".join(graph_parts)
    else:
        extra = ""

    # Step 4: assemble context
    ctx_parts = []
    for _, row in hits.iterrows():
        ctx_parts.append(f"[SOURCE: {row['paper_id']} | {row['chunk_id']}]\n{row['text']}")
    ctx = "\n\n---\n\n".join(ctx_parts) + extra

    # Step 5: generate answer
    gen = client.chat.completions.create(
        model='gpt-4.1',
        messages=[{"role": "user", "content": (
            "You are a scientific research assistant. Answer the question using ONLY the source passages below.\n\n"
            "Instructions:\n"
            "1. Answer using ONLY the literal text of the passages below. Do not use any knowledge "
            "about these papers or topics from your training data — even if you recognise the paper titles.\n"
            "2. Every sentence must be directly traceable to a word or phrase in one of the passages. "
            "If a detail is not stated in a passage, do not include it.\n"
            "3. Cite each source after claims using its label, e.g. [SOURCE: paper_id | chunk_id].\n"
            "4. If the answer cannot be found in the sources, say so explicitly.\n\n"
            f"--- SOURCES ---\n{ctx}\n--- END SOURCES ---\n\n"
            f"Question: {question}\n\nAnswer:"
        )}]
    )
    answer = gen.choices[0].message.content.strip()

    print(f"Q: {question}\n")
    print(answer)
    print("\n" + "=" * 70 + "\n")


questions = [
    "What strategies are used to reduce false positive gene fusion calls?",
    "How are detected fusion transcripts validated in a clinical setting?",
    "What is the significance of healthy tissue databases in fusion transcript research?",
    "Which computational tools are compared for RNA-seq based fusion detection, and how do they differ?",
]

for q in questions:
    ask(q)

---

## What We Built

This notebook implements a **graph-augmented RAG pipeline** over a personal corpus of scientific papers. Starting from raw PDFs, it builds two complementary indices:

- A **vector index** (sentence embeddings + cosine similarity) for semantic retrieval
- A **knowledge graph** (extracted SPO triplets, filtered and normalised) for relational expansion

At query time, these two indices work in sequence: vector search finds the most semantically relevant passages, then graph traversal expands outward to related concepts the embedding similarity alone would miss. The evaluation section measures whether that expansion was faithful and substantive.

## What This Is Not

This is a **lightweight educational implementation**, not a production GraphRAG system. Key differences from systems like Microsoft's GraphRAG:

| This notebook | Microsoft GraphRAG |
|---|---|
| Graph built from extracted triplets | Graph built from LLM-summarised communities |
| Entity resolution via single LLM call | Leiden algorithm for community detection |
| Graph expansion via string matching | Graph communities used as retrieval units |
| Evaluates individual queries | Designed for global corpus-level questions |

## Natural Next Steps

If you wanted to take this further:

- **Vector store**: replace the in-memory numpy array with FAISS or ChromaDB for sub-millisecond retrieval at scale
- **Entity resolution**: replace the single-call normalisation with a dedicated NER model (e.g. spaCy with a biomedical model) for more robust entity extraction
- **Community detection**: run the Leiden algorithm over the graph to discover thematic clusters, then use cluster summaries as an additional retrieval layer — this is the step that bridges this implementation and full GraphRAG
- **Multi-query evaluation**: run the evaluation suite over a set of queries rather than one, and aggregate faithfulness scores to get a stable picture of pipeline quality
- **Reranking**: add a cross-encoder reranker between retrieval and generation to improve passage selection precision
- **RDF and SPARQL**: migrate the knowledge graph from NetworkX to an RDF triplestore (e.g. `rdflib` for in-process use, or Apache Jena for a persistent server). The SPO triplets this pipeline produces map directly onto RDF triples, so the conversion is straightforward. The payoff is SPARQL — a full declarative query language — and the ability to federate your extracted graph against external biomedical ontologies such as HPO, OMIM, or the Sequence Ontology. This is the natural next step if you want to ask questions like "find all entities that are a subclass of rare disease as defined by ORDO" rather than simple neighbourhood traversal.
- **Natural language graph queries**: add a "talk to your graph" interface by having an LLM translate a natural language question into a graph query and execute it. Two implementation paths exist depending on how far you want to go. The lightweight option is LLM-generated Python: the model produces `nx.*` calls against the existing NetworkX graph, which works well for simple traversal and degree questions but is fragile for complex pattern matching. The production option is to export the graph to Neo4j and use LangChain's `GraphCypherQAChain`, which translates questions into Cypher — Neo4j's declarative query language — and handles multi-hop pattern matching cleanly. The SPO structure this pipeline produces exports to Neo4j trivially, so the migration path is straightforward when the query complexity warrants it.
